[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/QuantLet/NN_DL/blob/main/NN_DL_ch06/NN_DL_ch06_AttentionViz/NN_DL_ch06_AttentionViz.ipynb)

# NN_DL_ch06_AttentionViz

**Attention Mechanism & Positional Encoding from Scratch**

Implement scaled dot-product attention and positional encoding. Visualize attention weights and encoding patterns.

In [ ]:
!pip install -q torch matplotlib

In [ ]:
# Style & Reproducibility
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

MAIN_BLUE = '#1A3A6E'
IDA_RED   = '#CD0000'
FOREST    = '#2E7D32'
CRIMSON   = '#DC3545'
AMBER     = '#FFC107'

plt.rcParams.update({
    'figure.facecolor': 'white', 'axes.facecolor': 'white',
    'axes.grid': True, 'grid.alpha': 0.3,
    'font.size': 12, 'axes.labelsize': 13, 'axes.titlesize': 14,
    'figure.figsize': (10, 6),
})
np.random.seed(42)

def save_fig(name):
    plt.savefig(f'{name}.pdf', bbox_inches='tight', dpi=300, transparent=True)
    plt.savefig(f'{name}.png', bbox_inches='tight', dpi=150, transparent=True)
    plt.show()
    print(f'Saved {name}.pdf and {name}.png')

In [ ]:
import torch
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

In [ ]:
# Scaled Dot-Product Attention from Scratch
import torch
import torch.nn.functional as F

def scaled_dot_product_attention(Q, K, V, mask=None):
    d_k = Q.size(-1)
    scores = torch.matmul(Q, K.transpose(-2, -1)) / np.sqrt(d_k)
    if mask is not None:
        scores = scores.masked_fill(mask == 0, -1e9)
    weights = F.softmax(scores, dim=-1)
    output = torch.matmul(weights, V)
    return output, weights

torch.manual_seed(42)
seq_len, d_k = 8, 16
Q = torch.randn(1, seq_len, d_k)
K = torch.randn(1, seq_len, d_k)
V = torch.randn(1, seq_len, d_k)

out, attn_weights = scaled_dot_product_attention(Q, K, V)
print(f"Output shape: {out.shape}")
print(f"Attention weights shape: {attn_weights.shape}")

causal_mask = torch.tril(torch.ones(seq_len, seq_len)).unsqueeze(0)
out_causal, attn_causal = scaled_dot_product_attention(Q, K, V, mask=causal_mask)

In [ ]:
# Visualize Attention Weights
tokens = ['The', 'stock', 'market', 'crashed', 'after', 'the', 'Fed', 'announcement']

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

im1 = axes[0].imshow(attn_weights.squeeze().detach().numpy(), cmap='YlOrRd', aspect='auto')
axes[0].set_xticks(range(seq_len)); axes[0].set_xticklabels(tokens, rotation=45, ha='right')
axes[0].set_yticks(range(seq_len)); axes[0].set_yticklabels(tokens)
axes[0].set_title('Full Self-Attention', fontweight='bold')
plt.colorbar(im1, ax=axes[0])

im2 = axes[1].imshow(attn_causal.squeeze().detach().numpy(), cmap='YlOrRd', aspect='auto')
axes[1].set_xticks(range(seq_len)); axes[1].set_xticklabels(tokens, rotation=45, ha='right')
axes[1].set_yticks(range(seq_len)); axes[1].set_yticklabels(tokens)
axes[1].set_title('Causal (Masked) Self-Attention', fontweight='bold')
plt.colorbar(im2, ax=axes[1])

plt.suptitle('Scaled Dot-Product Attention', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
save_fig('nn_ch06_attention_mechanism')

In [ ]:
# Positional Encoding Visualization

def positional_encoding(max_len, d_model):
    pe = np.zeros((max_len, d_model))
    position = np.arange(max_len)[:, np.newaxis]
    div_term = np.exp(np.arange(0, d_model, 2) * -(np.log(10000.0) / d_model))
    pe[:, 0::2] = np.sin(position * div_term)
    pe[:, 1::2] = np.cos(position * div_term)
    return pe

max_len, d_model = 100, 64
pe = positional_encoding(max_len, d_model)

fig, axes = plt.subplots(2, 2, figsize=(16, 10))

im = axes[0, 0].imshow(pe, cmap='RdBu_r', aspect='auto')
axes[0, 0].set_xlabel('Embedding Dimension'); axes[0, 0].set_ylabel('Position')
axes[0, 0].set_title('Positional Encoding Matrix', fontweight='bold')
plt.colorbar(im, ax=axes[0, 0])

for dim, color in zip([0, 1, 10, 20], [MAIN_BLUE, IDA_RED, FOREST, AMBER]):
    axes[0, 1].plot(pe[:, dim], color=color, lw=1.5, label=f'dim={dim}')
axes[0, 1].set_xlabel('Position'); axes[0, 1].set_ylabel('Value')
axes[0, 1].set_title('PE Values for Selected Dimensions', fontweight='bold')
axes[0, 1].legend()

from numpy.linalg import norm
pe_norm = pe / (norm(pe, axis=1, keepdims=True) + 1e-10)
cos_sim = pe_norm @ pe_norm.T
im2 = axes[1, 0].imshow(cos_sim[:50, :50], cmap='RdBu_r', aspect='auto', vmin=-1, vmax=1)
axes[1, 0].set_xlabel('Position j'); axes[1, 0].set_ylabel('Position i')
axes[1, 0].set_title('Cosine Similarity Between Positions', fontweight='bold')
plt.colorbar(im2, ax=axes[1, 0])

axes[1, 1].bar(range(0, d_model, 2),
               1 / (10000 ** (np.arange(0, d_model, 2) / d_model)),
               color=MAIN_BLUE, edgecolor='white')
axes[1, 1].set_xlabel('Dimension (even)'); axes[1, 1].set_ylabel('Frequency')
axes[1, 1].set_title('PE Frequency per Dimension', fontweight='bold')
axes[1, 1].set_yscale('log')

plt.suptitle('Positional Encoding in Transformers', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
save_fig('nn_ch06_positional_encoding')

## Summary

Implemented **scaled dot-product attention** from scratch and visualized full vs. causal attention patterns. Created comprehensive **positional encoding** visualizations showing sinusoidal patterns, position similarities, and frequency spectrum.